# Subnetwork Extraction and Model Mask Generation Demo

This notebook demonstrates how to:
1. Extract a clean, duplicate-free list of upstream reach IDs starting from any arbitrary segment (for use as a mask in `t-route` execution) using the new `get_upstream_mask()` utility.
2. Visualize topological structure (DAG) of selected subgraphs using the recursive ASCII tree printer.

These tasks support diagnostic subsetting of stream channel networks from NWM domain data.

### 1. Setup Environment and Load Data
First, we set up repository paths and import the `troute_network` package modules.

In [ ]:
import os
import sys

# Set high recursion limit for deep networks
sys.setrecursionlimit(6000)

# Resolve path to the test data directory (one level up from notebooks/)
root = os.path.dirname(os.path.abspath(""))
geo_input_folder = os.path.join(root, "test_data")

import troute_network.nhd_network_utilities as nnu
import troute_network.recursive_print as rp
from troute_network import get_upstream_mask

print("Environment ready.")

In [ ]:
# Load the Brazos & Lower Colorado River NHD dataset (subset included in this repository)
print("Loading Brazos & Lower Colorado network...")
data, values = nnu.set_networks(
    supernetwork="Brazos_LowerColorado_ge5",
    geo_input_folder=geo_input_folder,
    verbose=False,
    debuglevel=-1
)

connections = values[0]
terminal_keys = values[4]
circular_keys = values[6]
terminal_keys_super = terminal_keys - circular_keys
terminal_code = data["terminal_code"]

print(f"Loaded network with {len(connections)} total segments.")
print(f"Found {len(terminal_keys_super)} independent subnetworks.")

### 2. Extract a Clean Upstream Reach Mask
Here we select a single subnetwork (represented by its terminal outlet ID) and extract its entire upstream drainage mask. We then save this list of unique reach IDs directly to a text file. This text file can be directly used as a mask file for model execution in `t-route`.

In [ ]:
# Choose a subnetwork terminal outlet key
target_outlet = sorted(list(terminal_keys_super))[0]
print(f"Selected Outlet Reach ID: {target_outlet}")

# Extract the mask of all reach IDs upstream of this outlet
upstream_mask = get_upstream_mask(target_outlet, connections, terminal_code)

print(f"Total reaches in subnetwork: {len(upstream_mask)}")
print(f"First 15 reach IDs in subnetwork: {sorted(list(upstream_mask))[:15]}")

In [ ]:
# Save the clean reach list to a text file (one ID per line)
mask_filename = os.path.join(root, "test_data", f"mask_{target_outlet}.txt")
with open(mask_filename, "w") as f:
    for reach_id in sorted(upstream_mask):
        f.write(f"{reach_id}\n")

print(f"Saved clean mask file to: {mask_filename}")
print("This file is now ready for use in model configurations.")

### 3. Generate ASCII Tree Representations for a Selected Subgraph
We can isolate a specific subgraph (e.g., Walnut Creek, Mulberry Creek, or Cahaba River) and print its ASCII representation without showing the other networks.

In [ ]:
# Print the ASCII tree structure for our selected subnetwork
print(f"Topological tree structure starting from outlet {target_outlet}:")
rp.print_connections(
    terminal_keys={target_outlet},
    up_connections=connections,
    down_connections=connections,
    terminal_code=terminal_code
)